# SQLite Database Inspector

Use this notebook to inspect the API's SQLite database directly.

**Prerequisites:** The API container must have been started at least once so `./data/db/labelling.db` exists.

Run this notebook from the `on_premise_deployment/` folder, or adjust `DB_PATH` below.

In [ ]:
import sqlite3
import pandas as pd

# Path to the mounted database — adjust if your LOCAL_DATA_DIR is different
DB_PATH = ".." #location of the database file

con = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
print("Tables:", tables["name"].tolist())

## API Keys
Keys created by `docker exec ops-risk-api python seed.py`.

In [ ]:
df = pd.read_sql("SELECT * FROM api_keys ORDER BY created_at DESC", con)
print(f"{len(df)} key(s)")
df

## Sessions
One row per session ever created. Rows persist even after a session expires or is deleted.

In [ ]:
df = pd.read_sql("SELECT * FROM sessions ORDER BY created_at DESC", con)
print(f"{len(df)} session(s)")
df

## Labelling History
Written when a labelling pipeline run completes.

In [ ]:
df = pd.read_sql("SELECT * FROM labelling_history ORDER BY completed_at DESC", con)
print(f"{len(df)} completed run(s)")
df

## Risk Score History
Written when a risk score generation run completes.

In [ ]:
df = pd.read_sql("SELECT * FROM risk_score_history ORDER BY completed_at DESC", con)
print(f"{len(df)} completed run(s)")
df

## LLM Usage
Token consumption per LLM call — useful for cost tracking.

In [ ]:
df = pd.read_sql("SELECT * FROM llm_usage ORDER BY called_at DESC", con)
print(f"{len(df)} LLM call(s)")
df

In [ ]:
# Total token usage summary
if len(df) > 0:
    summary = df[["tokens_in", "tokens_out", "cached_tokens"]].sum()
    print("Total token usage:")
    print(summary.to_string())

## Step Timing
Duration of each pipeline step per session — useful for performance debugging.

In [ ]:
df = pd.read_sql("SELECT * FROM step_timing ORDER BY started_at DESC", con)
print(f"{len(df)} step record(s)")
df

In [ ]:
# Total time per session
if len(df) > 0:
    summary = df.groupby(["auth_id", "pipeline"])["duration_sec"].sum().reset_index()
    summary.columns = ["auth_id", "pipeline", "total_duration_sec"]
    print(summary)

## Organisations

In [ ]:
df = pd.read_sql("SELECT * FROM organisations", con)
df

## Run any custom SQL

In [ ]:
query = """
SELECT s.auth_id, s.name, s.status, s.created_at, l.completed_at
FROM sessions s
LEFT JOIN labelling_history l ON s.auth_id = l.auth_id
ORDER BY s.created_at DESC
"""
pd.read_sql(query, con)